# Bootstrap Few-shot Prompting with LangSmith

Prompt engineering is a pain. You can use _examples_ to optimize the prompt for you with the help of tools like LangSmith. Instead of guessing which examples will be the most impactful, you can use tried-and-true evaluation practices to curate and compile the right examples for your pipeline. The main steps are:

1. Create a dataset
2. Pick a metric to improve
3. Create an initial system
4. Decide the update logic (few-shot examples vs. instruction teaching vs. other methods, how to format the examples, etc.)
5. Train!


Below is an example bootstrapping a gpt-3.5-turbo model on an entailment task using few-shot examples. This example inspired by Christopher Potts' [example](https://github.com/stanfordnlp/dspy/blob/main/examples/nli/scone/scone.ipynb) on the SCONE dataset.

The task is natural language inference, where the LLM is required to predict whether the a statement can be logically concluded from a premise / grounding statement.

In [1]:
# Dependencies already installed

In [2]:
import os
import time
import truststore

# Use macOS system trust store (includes corporate CA certs)
truststore.inject_into_ssl()

os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [3]:
# We can do the same thing with a SQLite cache
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

set_llm_cache(SQLiteCache(database_path=".langchain.db"))

In [4]:
from langsmith import Client

client = Client()

public_datasets = [
    "https://smith.langchain.com/public/1d065de2-56c1-496e-bc66-bdce308e6537/d",  # train
    "https://smith.langchain.com/public/3205fa05-bd78-4eaf-924f-96df0f577b1f/d",  # train2
    "https://smith.langchain.com/public/fdf16166-1edd-418f-b777-3af82034931d/d",  # dev
    "https://smith.langchain.com/public/aee61506-3c60-4ca8-95c4-0314c9719ca8/d",  # dev2
    "https://smith.langchain.com/public/8d40d210-f8e6-4def-a206-78c5080c5d53/d",  # test
]
for ds in public_datasets:
    try:
        client.clone_public_dataset(ds)
    except Exception:
        pass  # Already cloned

train_name = "scone-train2"
dev_name = "scone-dev2"
test_name = "scone-test-one-scoped"
full_test_name = "scone-test"

example = next(client.list_examples(dataset_name=train_name))
print("inputs", example.inputs)
print("outputs", example.outputs)

inputs {'context': 'There is not a single person walking in the city.', 'question': 'Can we logically conclude for sure that there is not a single mover walking in the city?'}
outputs {'answer': 'Yes', 'category': 'one_scoped'}


Reviewing the values above, these examples can be tricky! 

## Evaluator

Since we have ground-truth clasification labels, we can use an exact-match criterion as our evaluator.

In [5]:
from langsmith.schemas import Run, Example


def exact_match(run: Run, example: Example) -> dict:
    """Evaluate exact match correctness of the NLI result."""
    try:
        predicted = run.outputs["is_entailed"]
        expected = example.outputs["answer"]
        score = expected.lower() == predicted.lower()
    except Exception:
        try:
            expected = example.outputs["answer"]
            expected_bool = {"no": False, "yes": True}.get(expected.strip().lower())
            score = run.outputs["output"].is_entailed == expected_bool
        except Exception:
            score = 0
    return {"key": "exact_match", "score": int(score)}

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Groq LLM via OpenAI-compatible API
groq_llm = ChatOpenAI(
    model="llama-3.1-8b-instant",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
    temperature=0,
)

# Prompt with placeholder for few-shot examples
prompt = PromptTemplate.from_template(
    """You are given some context (a premise) and a question (a hypothesis). You must indicate with Yes/No answer whether we can logically conclude the hypothesis from the premise.

---

Follow the following format.

Context: ${{context}}

Question: ${{question}}

Reasoning: Let's think step by step in order to ${{produce the answer}}. We ...

Answer: Yes or No

---{examples}

Context: {context}

Question: {question}

Reasoning: Let's think step by step in order to"""
).partial(examples="")


def parse(pred: str):
    fnd = "\nAnswer:"
    idx = pred.find(fnd)
    if idx == -1:
        # Try to extract Yes/No from the text
        lower = pred.lower().strip()
        answer = "Yes" if "yes" in lower.split()[-3:] else "No"
        return {"is_entailed": answer, "reasoning": pred.strip()}
    answer = pred[idx + len(fnd) :].strip()
    return {"is_entailed": answer, "reasoning": pred[:idx].strip()}


chain = prompt | groq_llm | StrOutputParser() | parse

In [7]:
time.sleep(3)
prediction = chain.invoke(example.inputs)
prediction

{'is_entailed': 'No',
 'reasoning': "Context: There is not a single person walking in the city.\n\nQuestion: Can we logically conclude for sure that there is not a single mover walking in the city?\n\nReasoning: Let's think step by step in order to determine if we can conclude the hypothesis from the premise. We know that a person is a type of entity that can be a mover, but we don't have any information about the movers in the city. The premise only talks about people, not movers. Therefore, we cannot logically conclude that there is not a single mover walking in the city from the given premise."}

## Initial Evaluation

In [8]:
from langsmith.evaluation import evaluate

# Quick test: evaluate on dev set
print("Evaluator ready")

Evaluator ready


In [9]:
def predict_nli(inputs: dict) -> dict:
    time.sleep(3)
    return chain.invoke(inputs)

initial_results = evaluate(
    predict_nli,
    data=dev_name,
    evaluators=[exact_match],
    experiment_prefix="scone-initial",
    max_concurrency=1,
)
print(f"Initial evaluation: {initial_results.experiment_name}")

View the evaluation results for experiment: 'scone-initial-034d06ba' at:
https://smith.langchain.com/o/1ffcb2ed-904b-4ab3-885e-5e23637ec0c3/datasets/0f359887-b08e-4031-9420-918c3ee0d6cf/compare?selectedSessions=9583c292-6f8f-4168-a592-21315aa0df86




0it [00:00, ?it/s]

Initial evaluation: scone-initial-034d06ba


Got about ~55% on it. Definitely room for improvement.

## ✨ Optimize ✨


This just means to "use data to update the system". At present, LangChain runnables don't natively support a "backwards" method (a la pytorch), but you can pretty easily define updates/mutations for key important components you'd want to update, (such as prompts or LLMs).

For instance, component-wise, you could apply:
- Few shot prompting: add an additional string input or MessagesPlaceholder in the prompt template
- Updating the instructions: update the prompt template directly (likely the system prompt)
- LLM: do a backwards pass.

We will focus on few-shot prompting to limit the search space. We will then apply a genetic/evolutionary algorithm to compare performance of different few-shot examples and pick the ones that provide the most "lift" of the provided metric.

We'll first create a constructor for our chain that accepts the few-shot examples, letting us re-create the chain with each updated state.

In [10]:
import random
from typing import List, Optional

from langchain_core.runnables import RunnableLambda


def format_example(example: dict):
    inputs = example["input"]
    outputs = example["output"]
    return f"""

Context: {inputs['context']}

Question: {inputs['question']}

Reasoning: {outputs['reasoning']}

Answer: {outputs['is_entailed']}

"""


def format_few_shot(input_: dict, examples: Optional[List[dict]] = None):
    if examples:
        input_["examples"] = (
            "--".join(format_example(e) for i, e in enumerate(examples)) + "--"
        )
    return input_


def create_chain(examples: Optional[List] = None, llm=None):
    llm = llm or groq_llm
    chain = (
        RunnableLambda(format_few_shot).bind(examples=examples)
        | prompt
        | llm
        | StrOutputParser()
        | parse
    ).with_config(tags=["to_train"])
    return chain

## Training

Next, we'll define the training utilities.

In [11]:
from langchain_core.tracers.context import collect_runs


def step(
    construct_chain,
    train_examples,
    examples=None,
    bootstrap_k: int = 4,
):
    collected = examples.copy() if examples else []
    random.shuffle(train_examples)
    train_examples = train_examples.copy()
    while train_examples:
        if len(collected) >= bootstrap_k:
            break
        remaining = bootstrap_k - len(collected)
        train_batch = [train_examples.pop() for _ in range(min(remaining, len(train_examples)))]
        chain = construct_chain([e for e in collected if e.get("id") != example.id])
        # Sequential calls with rate limiting (instead of batch)
        for ex in train_batch:
            time.sleep(3)
            try:
                with collect_runs() as cb:
                    chain.invoke(ex.inputs)
                run = cb.traced_runs[0]
                # Manual evaluation
                try:
                    predicted = run.outputs["is_entailed"]
                    expected = ex.outputs["answer"]
                    score = expected.lower() == predicted.lower()
                except Exception:
                    score = False
                if score:
                    collected.append({
                        "input": ex.inputs,
                        "output": run.outputs,
                        "id": ex.id,
                    })
            except Exception as e:
                print(f"  Error processing example: {e}")
                time.sleep(5)
    return collected


def eval_step(eval_dataset, chain, step_n) -> float:
    """Compute the metrics on the validation dataset."""
    def predict(inputs: dict) -> dict:
        time.sleep(3)
        return chain.invoke(inputs)

    res = evaluate(
        predict,
        data=eval_dataset,
        evaluators=[exact_match],
        experiment_prefix=f"scone-step-{step_n}",
        max_concurrency=1,
    )
    # Aggregate scores from the runs
    runs = list(client.list_runs(project_name=res.experiment_name, execution_order=1))
    feedbacks = list(client.list_feedback(run_ids=[r.id for r in runs]))
    scores = [f.score for f in feedbacks if f.key == "exact_match" and f.score is not None]
    mean_score = sum(scores) / len(scores) if scores else 0
    print(f"  Step {step_n}: {mean_score:.2%} ({len(scores)} examples)")
    return mean_score


def train(
    chain_constructor,
    train_dataset,
    eval_dataset,
    steps: int = 3,
    k: int = 4,
    bootstrap_k: int = 4,
):
    """Run the full training loop."""
    print("Running initial evaluation...")
    best_score = eval_step(eval_dataset, chain_constructor(), 0)
    best_step = 0
    scores = [(best_score, [])]
    train_examples = list(client.list_examples(dataset_name=train_dataset))
    print(f"Training with {len(train_examples)} examples, {steps} steps, k={k}")

    for step_number in range(steps):
        print(f"\n--- Step {step_number + 1} ---")
        collected = step(
            chain_constructor, train_examples, bootstrap_k=bootstrap_k
        )
        if len(collected) < k:
            to_sample = min(k - len(collected), len(train_examples))
            collected += random.sample(train_examples, to_sample)
        selected_examples = collected
        print(f"  Collected {len(selected_examples)} examples")
        updated_chain = chain_constructor(examples=selected_examples)
        updated_score = eval_step(eval_dataset, updated_chain, step_number + 1)
        scores.append((updated_score, selected_examples))

        if updated_score > best_score:
            print(f"  New best score {updated_score:.2%} > {best_score:.2%}!")
            best_score = updated_score
            best_step = step_number + 1
        else:
            print(f"  Underperformed ({updated_score:.2%} <= {best_score:.2%}). Continuing.")

    print(f"\nBest overall score: {best_score:.2%}")
    print(f"Best step: {best_step}")
    return sorted(scores, key=lambda x: x[0], reverse=True)

#### Train

Now we can finally run the training loop!

In [12]:
import functools

# Train with Groq llama (reduced steps for rate limits)
all_scores = train(
    functools.partial(create_chain, llm=groq_llm),
    train_name,
    dev_name,
    steps=3,
    k=4,
    bootstrap_k=4,
)

Running initial evaluation...
View the evaluation results for experiment: 'scone-step-0-9d844fb9' at:
https://smith.langchain.com/o/1ffcb2ed-904b-4ab3-885e-5e23637ec0c3/datasets/0f359887-b08e-4031-9420-918c3ee0d6cf/compare?selectedSessions=f9204013-cf37-4e8a-9828-04e052edc360




0it [00:00, ?it/s]

  Step 0: 42.86% (49 examples)
Training with 200 examples, 3 steps, k=4

--- Step 1 ---
  Collected 4 examples
View the evaluation results for experiment: 'scone-step-1-8d6ed1fe' at:
https://smith.langchain.com/o/1ffcb2ed-904b-4ab3-885e-5e23637ec0c3/datasets/0f359887-b08e-4031-9420-918c3ee0d6cf/compare?selectedSessions=f56a68d9-f69f-4d9e-8006-6cf9fc166b90




0it [00:00, ?it/s]

  Step 1: 54.00% (50 examples)
  New best score 54.00% > 42.86%!

--- Step 2 ---
  Collected 4 examples
View the evaluation results for experiment: 'scone-step-2-ffe25af3' at:
https://smith.langchain.com/o/1ffcb2ed-904b-4ab3-885e-5e23637ec0c3/datasets/0f359887-b08e-4031-9420-918c3ee0d6cf/compare?selectedSessions=43022046-1373-4e66-acb5-fabf3d243a8f




0it [00:00, ?it/s]

  Step 2: 55.10% (49 examples)
  New best score 55.10% > 54.00%!

--- Step 3 ---
  Collected 4 examples
View the evaluation results for experiment: 'scone-step-3-039e6789' at:
https://smith.langchain.com/o/1ffcb2ed-904b-4ab3-885e-5e23637ec0c3/datasets/0f359887-b08e-4031-9420-918c3ee0d6cf/compare?selectedSessions=f0e01850-8a42-4284-b662-a02012cebcef




0it [00:00, ?it/s]

  Step 3: 48.00% (50 examples)
  Underperformed (48.00% <= 55.10%). Continuing.

Best overall score: 55.10%
Best step: 2


## Compare on held-out set

It's easy to overfit a single benchmark if you explicitly choose your pipeline based on metrics on that benchmark.

Let's compare models on an unseen test set to see whether the selected examples are reliably better.

In [13]:
best_score, best_examples = all_scores[0]

In [14]:
original_model = create_chain()
# This time we will apply gpt-3.5-turbo, but use the few-shot examples + reasoning trajectories
# from gpt-4 to help induce better performance
best_performing_model = create_chain(best_examples)

In [ ]:
# Check which test datasets are available
available = [ds.name for ds in client.list_datasets() if "scone" in ds.name.lower()]
print("Available scone datasets:", available)

# Use whichever test set is available
actual_test = test_name if test_name in available else (full_test_name if full_test_name in available else dev_name)
print(f"Using test dataset: {actual_test}")

def predict_optimized(inputs: dict) -> dict:
    time.sleep(3)
    return best_performing_model.invoke(inputs)

res_test = evaluate(
    predict_optimized,
    data=actual_test,
    evaluators=[exact_match],
    experiment_prefix="scone-optimized-test",
    max_concurrency=1,
)
print(f"Test evaluation: {res_test.experiment_name}")

Available scone datasets: ['scone-dev2', 'scone-dev-dataset', 'scone-test', 'scone-train', 'scone-train2']
Using test dataset: scone-test
View the evaluation results for experiment: 'scone-optimized-test-9e1525ba' at:
https://smith.langchain.com/o/1ffcb2ed-904b-4ab3-885e-5e23637ec0c3/datasets/7b70fda0-e289-4524-9cba-f8ca679e64ff/compare?selectedSessions=f1037496-7edb-4a2e-9771-5c78d89944e0




0it [00:00, ?it/s]

Error running target function: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01hyt2qjsxexkvrt6rcp9yvad4` service tier `on_demand` on tokens per day (TPD): Limit 500000, Used 499756, Requested 865. Please try again in 1m47.3088s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Traceback (most recent call last):
  File "/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/lc-academy-env/lib/python3.12/site-packages/langsmith/evaluation/_runner.py", line 1903, in _forward
    fn(*args, langsmith_extra=langsmith_extra)
  File "/var/folders/wr/xj2yx28s7xvg8trnrb4nc5_w0000gn/T/ipykernel_62860/1680590848.py", line 11, in predict_optimized
    return best_performing_model.invoke(inputs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/L107127/Library/CloudSt

Using the GPT-4 generated examples, we were able to boost the performance from ~0.54 to ~0.87: not bad!